In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgnet.download import download_collection
from imgtools.nifti.crawl import Crawler
from rich import print

/home/joshua-siraj/Documents/BHKLAB/med-image_index/.pixi/envs/default/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])

[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver',
    'BREAST-DIAGNOSIS'
]

In [3]:
source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="7957516",
    filenames=["nifti_and_segms.zip", "LiverHccSeg_MetaData.xlsx"],
    post_download=["unzip"]
)

In [8]:
dataset_index_path = (indexed_datasets.imgtools_path / "LiverHccSeg")
dataset_index_path.mkdir(parents=True, exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [4]:
print(indexed_datasets.source_config("LiverHccSeg"))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='zenodo',
    record_id='7957516',
    filenames=['nifti_and_segms.zip', 'LiverHccSeg_MetaData.xlsx'],
    post_download=['unzip']
)

In [10]:
temp_data_path = Path("temp_data") / "LiverHccSeg"
temp_data_path.mkdir(parents=True, exist_ok=True)
download_collection("LiverHccSeg", temp_data_path, indexed_datasets)

11:26:58 [info     ] Downloading file 'nifti_and_segms.zip' from Zenodo record 7957516 [imagenet] call=dispatcher._download_zenodo:55
nifti_and_segms.zip: 100%|██████████| 978M/978M [13:24<00:00, 1.21MB/s]
11:40:25 [info     ] Downloading file 'LiverHccSeg_MetaData.xlsx' from Zenodo record 7957516 [imagenet] call=dispatcher._download_zenodo:55
LiverHccSeg_MetaData.xlsx: 100%|██████████| 12.9k/12.9k [00:00<00:00, 84.5kB/s]
11:40:26 [info     ] Unpacking nifti_and_segms.zip  [imagenet] call=dispatcher._post_unzip:65


In [4]:
import pandas as pd
import re

temp_data_path = Path("temp_data") / "LiverHccSeg"
xlsx_path = temp_data_path / "LiverHccSeg_MetaData.xlsx"
csv_path = temp_data_path / "LiverHccSeg_MetaData.csv"
df = pd.read_excel(xlsx_path)
df = df.dropna(how="all")
# Flatten column names: Excel headers often have \r\n which break CSV parsing
df.columns = [re.sub(r"[\r\n\x0d_]+", " ", str(c)).strip() for c in df.columns]
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path}")
pd.read_csv(csv_path, encoding="utf-8-sig")

Wrote temp_data/LiverHccSeg/LiverHccSeg_MetaData.csv

,PatientID,StudyDate,Gender,Age,Ethiology,Contrast Agent,Contrast Agent Type,Contrast Agent Manufacturer,Magnetic Field Strength,Comments,Index,Location,Tumor Size,"APHE x000d (0=no, 1=yes)","Washout x000d (0=no, 1=yes)","Capsule x000d (0=no, 1=yes)",LI-RADS v2018,Portal Vein Thrombosis,"Ascitis on Imaging x000d (no=0, mild=1, severe=2)",Portal Hypertension x000d on Imaging
0,TCGA-BC-A3KG,2/2/02,F,68.0,NaN,Multihance,Hepatobiliary-specific,Bracco Diagnostics,1.5,NaN,tumor1,II/III,9.41,1.0,0.0,1.0,LR-5,no,0.0,no
1,TCGA-BC-A5W4,11/8/02,M,69.0,NaN,Multihance,Hepatobiliary-specific,Bracco Diagnostics,1.5,NaN,tumor1,"VI, VII",8.18,1.0,1.0,1.0,LR-5,no,0.0,Spelenomegaly
2,TCGA-BC-A69I,8/4/03,M,69.0,NaN,not avaliable,not avaliable,not avaliable,1.5,NaN,tumor1,IVA,6.20,1.0,0.0,1.0,LR-5,no,0.0,no
3,TCGA-DD-A4NF,2/24/04,M,71.0,NaN,not avaliable,not avaliable,not avaliable,1.5,NaN,tumor1,IVA,3.87,1.0,1.0,1.0,LR-5,no,0.0,no
4,TCGA-DD-A4NH,6/13/04,F,65.0,NaN,MultiHance,Hepatobiliary-specific,Bracco Diagnostics,1.5,NaN,tumor1,"III, IVB",11.04,1.0,1.0,0.0,"T-IV, definetively due to HCC",Tumor infiltrates left portal vein,0.0,no
5,TCGA-DD-A4NJ,3/30/04,F,54.0,NaN,MultiHance,Hepatobiliary-specific,Bracco Diagnostics,1.5,NaN,tumor1,VI,4.10,1.0,1.0,1.0,LR-5,no,0.0,no
6,TCGA-G3-A25T,6/26/01,F,44.0,NaN,Magnevist,Extracellular,Berlex,1.5,NaN,tumor1,"IVA, IVB,V, VIII",12.20,1.0,0.0,1.0,LR-M,no,0.0,no
7,TCGA-G3-A3CJ,11/24/04,M,52.0,HCV,Magnevist,Extracellular,Berlex,1.5,NaN,tumor1,"VII, VIII",12.30,1.0,1.0,1.0,LR-5,no,0.0,"Spelenomegaly, perisplenic varices"
8,TCGA-G3-A7M7,11/22/06,M,65.0,Alcohol,Gadovist,Extracellular,Schering,1.5,NaN,tumor1,V/VIII,3.96,1.0,1.0,1.0,LR-5,no,0.0,no
9,TCGA-G3-AAV1,1/25/07,M,51.0,"HBV, HCV",Gadovist,Extracellular,Schering,1.5,cysts and harmatomas,tumor1,VI,5.80,1.0,0.0,0.0,LR-M,no,0.0,no


In [12]:
import imgtools.loggers
import logging

imgtools.loggers.get_logger("imgtools").setLevel(logging.INFO)

crawler = Crawler(
    nifti_dir=temp_data_path / "nifti_and_segms",
    scan_name_pattern="{PatientID}/{StudyDate}/{PhaseContrast}.nii.gz",
    mask_name_pattern="{PatientID}/{StudyDate}/rater{rater_id}_{ROIName}.nii.gz",
    metadata_path=temp_data_path / "LiverHccSeg_MetaData.csv",
    metadata_join_col="PatientID",
    deep=True,
    output_dir=indexed_datasets.imgtools_path,
    dataset_name="LiverHccSeg",
    n_jobs=10,
    force=True,
)
crawler.crawl()




12:48:57 [info     ] Starting NIFTI crawl.          [imgtools] nifti_dir=PosixPath('temp_data/LiverHccSeg/nifti_and_segms') output_dir=PosixPath('indexed_datasets/.imgtools') dataset_name=LiverHccSeg call=crawler.crawl:92
         [info     ] Found 185 NIfTI files in /home/joshua-siraj/Documents/BHKLAB/med-image_index/temp_data/LiverHccSeg/nifti_and_segms. [imgtools] call=parse_niftis.parse_nifti_dir:462
         [info     ] Using shared keys: ['PatientID', 'StudyDate'] for reference_id, this will be used to link masks to their referenced scans [imgtools] call=parse_niftis.parse_nifti_dir:483
Parsing 185 files:   0%|          | 0/185 [00:00<?, ?it/s]12:48:57 [warning  ] File TCGA-BC-4073/02-21-2000/rater1_liver.nii.gz matched both scan and mask patterns. Returning mask match. [imgtools] call=parse_niftis._match_file:126
12:48:57 [warning  ] File TCGA-BC-4073/02-21-2000/rater2_liver.nii.gz matched both scan and mask patterns. Returning mask match. [imgtools] call=parse_niftis._match_fil

In [13]:
assert "LiverHccSeg" in indexed_datasets.collections

In [15]:
df = pd.read_csv(indexed_datasets.imgtools_path / "LiverHccSeg" / "index.csv")

# Add a 'Modality' column: 'SEG' for masks, 'MR' for scans
def modality_from_filetype(row):
    ft = str(row.get("file_type", "")).lower()
    if ft == "mask":
        return "SEG"
    elif ft == "scan":
        return "MR"
    return ""

df["Modality"] = df.apply(modality_from_filetype, axis=1)

In [16]:
# Use imgtools map to map column names to DICOM tags
import toml
with open(indexed_datasets.imgtools_path / "LiverHccSeg" / "index_mapping.toml", "r") as f:
    index_mapping = toml.load(f)

non_empty_mappings = {k: v for k, v in index_mapping.items() if v}
df.rename(columns=non_empty_mappings, inplace=True)
df.to_csv(indexed_datasets.imgtools_path / "LiverHccSeg" / "index.csv", index=False)
non_empty_mappings

{'StudyDate_x': 'StudyDate',
 'Gender': 'PatientSex',
 'Age': 'PatientAge',
 'Magnetic Field Strength': 'MagneticFieldStrength'}